# Using a pretrained quickdraw world model

Load a trained world model, hand it a few steps of context plus an action sequence, and watch it
**imagine** the frames that follow — open-loop, with no further observations.

Companion prose, including the traps: [`using_pretrained_models.md`](using_pretrained_models.md).

> `MODEL` below is a Hub repo id, or a local directory (e.g. a `+hub.dry_run=true` staging dir).


In [ ]:
MODEL = 'isaac-ronald-ward/quickdraw-wm-robocasa-vl128'   # or a local path
DEVICE = 'cuda'
HORIZON = 32          # imagined steps


## 1. Load

Three objects, all needed:

- `model` — the world model, already in `eval()` on your device
- `norm` — the **training** normalisation statistics, bundled in the repo
- `cfg` — the resolved training config (tells you `P`, the context length, and the head names)

The normaliser ships with the model on purpose: a checkpoint has empty `hyper_parameters`, and the
statistics normally live in the *dataset*, which for this model is private. If they are missing,
loading raises rather than guessing — wrong statistics would make every rollout silently wrong.


In [ ]:
from quickdraw import load_pretrained, load_example_context
import torch, numpy as np

model, norm, cfg = load_pretrained(MODEL, device=DEVICE)
ex = load_example_context(MODEL)   # real windows shipped in the repo; no dataset needed

P = int(cfg.data.P)
head = [k.split('frames__')[1] for k in ex if k.startswith('frames__')][0]
print(f'params      {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'context P   {P} steps')
print(f'image head  {head!r}')
print(f'example     obs {ex["obs"].shape}  act {ex["act"].shape}  frames {ex[f"frames__{head}"].shape}')


## 2. Imagine

Three things to get right, each of which fails quietly rather than loudly:

1. **Vectors are normalised, images are not.** `proprio` through `norm_obs`; frames are `[0,1]` floats.
2. **`acts` needs `P + HORIZON - 1` steps**, not `HORIZON` — the context consumes actions too.
3. **Pass `decode_chunk`** or the image decoder (~78% of per-sample memory) will exhaust the GPU.


In [ ]:
ctx = {'proprio': norm.norm_obs(torch.from_numpy(ex['obs'][:, :P])).float().to(DEVICE),
       head:      torch.from_numpy(ex[f'frames__{head}'][:, :P]).float().div(255).to(DEVICE)}
acts = norm.norm_act(torch.from_numpy(ex['act'][:, :P + HORIZON - 1])).float().to(DEVICE)

with torch.no_grad():
    out = model.imagine_eval(ctx, acts, horizon=HORIZON, decode_chunk=16)

pred    = out[head].clamp(0, 1)              # (B, H, HW, HW, 3) imagined frames
proprio = norm.denorm_obs(out['proprio'])    # (B, H, obs_dim) physical units
print('imagined frames ', tuple(pred.shape))
print('imagined proprio', tuple(proprio.shape), '| finite:', bool(torch.isfinite(proprio).all()))


## 3. Score it against ground truth

The example context carries the real continuation, so we can measure the rollout rather than just
look at it. PSNR here is over the whole imagined horizon from a single context — expect it to be
lower than a card's `@+128` number, which is averaged over many episodes under a fixed protocol.


In [ ]:
gt = torch.from_numpy(ex[f'frames__{head}'][:, P:P + HORIZON]).float().div(255).to(DEVICE)
mse = float(((pred - gt) ** 2).mean())
print(f'open-loop {HORIZON}-step   MSE {mse:.5f}   PSNR {-10*np.log10(mse):.2f} dB')

per_step = ((pred - gt) ** 2).mean(dim=(0, 2, 3, 4)).cpu().numpy()
for t in (0, HORIZON//4, HORIZON//2, HORIZON-1):
    print(f'  step {t:3d}: PSNR {-10*np.log10(per_step[t]):.2f} dB')


## 4. Look at it

The metric is not the point; the frames are. Top row is imagined, bottom row is truth.


In [ ]:
import matplotlib.pyplot as plt

ts = np.linspace(0, HORIZON - 1, 6).astype(int)
fig, ax = plt.subplots(2, len(ts), figsize=(2.0 * len(ts), 4.2))
for j, t in enumerate(ts):
    ax[0, j].imshow(pred[0, t].cpu().numpy()); ax[0, j].set_title(f'+{t+1}', fontsize=9)
    ax[1, j].imshow(gt[0, t].cpu().numpy())
    for i in (0, 1): ax[i, j].axis('off')
ax[0, 0].set_ylabel('imagined'); ax[1, 0].set_ylabel('truth')
fig.suptitle(f'open-loop imagination, {head}', y=0.99)
plt.tight_layout(); plt.show()


## 5. Long, cheap rollouts

Image decoding dominates the cost. If you only need states, skip it entirely — the latent rollout
still runs, you are just choosing what gets rendered.


In [ ]:
with torch.no_grad():
    long_out = model.imagine_eval(ctx, 
        norm.norm_act(torch.from_numpy(ex['act'][:, :P + 87 - 1])).float().to(DEVICE),
        horizon=87, heads=['proprio'])
print('proprio-only rollout:', tuple(long_out['proprio'].shape), '| image decoded:', head in long_out)
